# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamidrazabajwa49/flyrank-ml-internship-assignment-1/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Full signal-audit pass, sibling to `w04_baseline_score.ipynb` — same March-feature / April-label slice and lane, three signals instead of two, plus one flag-linked test taken further.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
%pip -q install duckdb
import os, getpass
import numpy as np
import pandas as pd
import duckdb
from scipy.stats import spearmanr

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = 'hf://datasets/FlyRank/internship-warehouse'
FACT_03 = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_04 = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{BASE}/dim_content.parquet')"
DECISION_DATE = pd.Timestamp('2026-03-31')

frame = con.sql(f"""
WITH mar AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS mar_impr, SUM(gsc_clicks) AS mar_clicks,
         SUM(gsc_sum_position) AS mar_sum_pos, COUNT(*) AS mar_days,
         SUM(gsc_impressions) FILTER (report_date <  DATE '2026-03-16') AS h1_impr,
         SUM(gsc_impressions) FILTER (report_date >= DATE '2026-03-16') AS h2_impr
  FROM {FACT_03} WHERE gsc_data_available IS TRUE GROUP BY 1, 2
),
apr AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS apr_impr, COUNT(*) AS apr_days
  FROM {FACT_04} WHERE gsc_data_available IS TRUE GROUP BY 1, 2
)
SELECT m.client_hash_id, m.content_hash_id, m.mar_impr, m.mar_clicks, m.mar_days,
       m.mar_impr * 1.0 / m.mar_days AS mar_daily_impr,
       m.mar_sum_pos * 1.0 / NULLIF(m.mar_impr, 0) AS mar_avg_position,
       m.mar_clicks * 100.0 / NULLIF(m.mar_impr, 0) AS mar_ctr,
       (m.h2_impr - m.h1_impr) * 1.0 / NULLIF(m.h2_impr + m.h1_impr, 0) AS mar_h2_vs_h1,
       a.apr_impr * 1.0 / NULLIF(a.apr_days, 0) AS apr_daily_impr_raw
FROM mar m LEFT JOIN apr a USING (client_hash_id, content_hash_id)
WHERE m.mar_days >= 15 AND m.mar_impr >= 30
""").df()

frame['mar_h2_vs_h1'] = frame['mar_h2_vs_h1'].fillna(0.0)
frame['apr_daily_impr'] = frame['apr_daily_impr_raw'].fillna(0.0)
frame['y'] = (frame['apr_daily_impr'] < 0.80 * frame['mar_daily_impr']).astype(int)
frame = frame.drop(columns=['apr_daily_impr_raw'])

content_meta = con.sql(f"SELECT content_hash_id, content_updated_date FROM {DIM_CONTENT}").df()
frame = frame.merge(content_meta, on='content_hash_id', how='left')
frame['days_since_update'] = (DECISION_DATE - pd.to_datetime(frame['content_updated_date'])).dt.days
frame['days_since_update'] = frame['days_since_update'].fillna(frame['days_since_update'].median())

BASE_RATE = frame['y'].mean()
print(f"frame: {len(frame):,} pages | {frame['client_hash_id'].nunique()} clients | base rate {BASE_RATE:.3f}")



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

frame: 116,539 pages | 40 clients | base rate 0.504


In [2]:
#  Distributions: mean vs median is the heavy-tail tell
for col in ['mar_daily_impr', 'mar_avg_position', 'mar_ctr', 'days_since_update']:
    s = frame[col]
    print(f"{col:20s} mean={s.mean():10.2f}  median={s.median():10.2f}  "
          f"p95={s.quantile(0.95):10.2f}  max={s.max():10.2f}  skew={s.skew():.2f}")

print("\nmar_daily_impr is the clearest case: mean sits well above the median whenever a few")
print("huge pages pull it up, with p95 far below max -- a heavy right tail, not a bell curve.")
print("Plain Pearson correlation on raw impressions would be dominated by those few giants,")
print("so every volume-based test below uses log1p(mar_daily_impr) or bucketed medians instead.")

frame['log_daily_impr'] = np.log1p(frame['mar_daily_impr'])
print(f"\nlog1p(mar_daily_impr): mean={frame['log_daily_impr'].mean():.2f}, "
      f"median={frame['log_daily_impr'].median():.2f}, skew={frame['log_daily_impr'].skew():.2f}  "
      "(skew much closer to 0 -- confirms the transform is doing its job)")


mar_daily_impr       mean=     79.04  median=     19.10  p95=    331.81  max=  21280.14  skew=17.57
mar_avg_position     mean=     15.91  median=      8.87  p95=     53.63  max=    106.89  skew=1.86
mar_ctr              mean=      0.26  median=      0.07  p95=      1.07  max=     16.28  skew=7.07
days_since_update    mean=    -49.60  median=    -50.00  p95=     34.00  max=    303.00  skew=1.11

mar_daily_impr is the clearest case: mean sits well above the median whenever a few
huge pages pull it up, with p95 far below max -- a heavy right tail, not a bell curve.
Plain Pearson correlation on raw impressions would be dominated by those few giants,
so every volume-based test below uses log1p(mar_daily_impr) or bucketed medians instead.

log1p(mar_daily_impr): mean=3.18, median=3.00, skew=0.51  (skew much closer to 0 -- confirms the transform is doing its job)


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Same verdict rule for all three: compare the extreme buckets, require a spread of at least 0.03 in decline rate to call it anything but FALSE, and refuse to verdict any bucket under the n=50 floor this skill sets.

In [3]:
MIN_N = 50

def verdict(table, rate_col='decline_rate', n_col='n', hypothesis='decreases', tol=0.03):
    if (table[n_col] < MIN_N).any():
        thin = table[table[n_col] < MIN_N]
        print(f"  ** {len(thin)} bucket(s) below n={MIN_N} -- excluded from the verdict, not just ignored **")
        table = table[table[n_col] >= MIN_N]
    if len(table) < 2:
        return 'INSUFFICIENT DATA', 0.0
    first, last = table[rate_col].iloc[0], table[rate_col].iloc[-1]
    spread = table[rate_col].max() - table[rate_col].min()
    if spread < tol:
        return 'FALSE', spread
    moved_as_hypothesized = (last < first) if hypothesis == 'decreases' else (last > first)
    return ('CONFIRMED' if moved_as_hypothesized else 'OPPOSITE'), spread

# Signal 1: worse position -> more decline?
frame['pos_bucket'] = pd.cut(frame['mar_avg_position'], bins=[0, 3, 10, 20, 50, 200],
                              labels=['1-3', '4-10', '11-20', '21-50', '50+'])
sig1 = frame.groupby('pos_bucket', observed=True).agg(
    n=('y', 'size'), decline_rate=('y', 'mean'), median_daily_impr=('mar_daily_impr', 'median')).round(3)
print('SIGNAL 1 -- claim: pages ranking worse decline more often')
print(sig1.to_string())
v1, s1 = verdict(sig1.reset_index(), hypothesis='increases')
print(f"VERDICT: {v1}  (spread {s1:.3f})\n")


SIGNAL 1 -- claim: pages ranking worse decline more often
                n  decline_rate  median_daily_impr
pos_bucket                                        
1-3         10355         0.583             50.500
4-10        52736         0.511             27.452
11-20       22546         0.501             14.806
21-50       23988         0.465             12.000
50+          6913         0.480              4.621
VERDICT: OPPOSITE  (spread 0.118)



In [4]:
# Signal 2: staleness -> more decline? (behind FlyRank's refresh flags)
frame['staleness_bucket'] = pd.qcut(frame['days_since_update'], 5,
                                     labels=['freshest', 'Q2', 'Q3', 'Q4', 'stalest'])
sig2 = frame.groupby('staleness_bucket', observed=True).agg(
    n=('y', 'size'), decline_rate=('y', 'mean'), median_days=('days_since_update', 'median')).round(3)
print('SIGNAL 2 -- claim: staler pages decline more often (the refresh-flag assumption)')
print(sig2.to_string())
v2, s2 = verdict(sig2.reset_index(), hypothesis='increases')
print(f"VERDICT: {v2}  (spread {s2:.3f})\n")


SIGNAL 2 -- claim: staler pages decline more often (the refresh-flag assumption)
                      n  decline_rate  median_days
staleness_bucket                                  
freshest          25336         0.624        -94.0
Q2                24699         0.427        -78.0
Q3                29509         0.485        -50.0
Q4                13802         0.421        -48.0
stalest           23193         0.529         34.0
VERDICT: OPPOSITE  (spread 0.203)



In [5]:
# Signal 3: volume -> less decline? (log1p, plus rank correlation, per heavy-tail handling)
frame['vol_bucket'] = pd.qcut(frame['log_daily_impr'], 5, labels=['Q1 low', 'Q2', 'Q3', 'Q4', 'Q5 high'])
sig3 = frame.groupby('vol_bucket', observed=True).agg(
    n=('y', 'size'), decline_rate=('y', 'mean'), median_daily_impr=('mar_daily_impr', 'median')).round(3)
print('SIGNAL 3 -- claim: higher-volume pages decline less often')
print(sig3.to_string())
v3, s3 = verdict(sig3.reset_index(), hypothesis='decreases')
print(f"VERDICT: {v3}  (spread {s3:.3f})")

rho, p = spearmanr(frame['log_daily_impr'], frame['y'])
print(f"Spearman rank correlation (log volume vs decline): rho={rho:.3f}, p={p:.4f}")
print("Reported alongside the bucket table, not instead of it -- a monotonic bucket trend and")
print("a matching correlation sign is a much stronger claim than either alone.")


SIGNAL 3 -- claim: higher-volume pages decline less often
                n  decline_rate  median_daily_impr
vol_bucket                                        
Q1 low      23311         0.437              2.957
Q2          23307         0.551              7.400
Q3          23322         0.561             19.097
Q4          23294         0.524             52.097
Q5 high     23305         0.449            186.862
VERDICT: OPPOSITE  (spread 0.124)
Spearman rank correlation (log volume vs decline): rho=0.003, p=0.3231
Reported alongside the bucket table, not instead of it -- a monotonic bucket trend and
a matching correlation sign is a much stronger claim than either alone.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Taking **staleness** further, since it sits directly behind the refresh flags: a shipped rule like "flag anything untouched for 180+ days" makes a specific, testable claim. Section 2 tested it as a smooth quintile trend; this section tests the actual threshold, and checks it isn't just volume wearing a staleness costume.

In [6]:
STALE_THRESHOLD = 180
stale = frame['days_since_update'] >= STALE_THRESHOLD

threshold_table = frame.groupby(stale.map({True: f'>= {STALE_THRESHOLD}d', False: f'< {STALE_THRESHOLD}d'})).agg(
    n=('y', 'size'), decline_rate=('y', 'mean'), median_daily_impr=('mar_daily_impr', 'median'))
print(f"Does the specific '{STALE_THRESHOLD}+ days stale' rule the refresh flag uses hold up?")
print(threshold_table.round(3).to_string())

gap = threshold_table.loc[f'>= {STALE_THRESHOLD}d', 'decline_rate'] - threshold_table.loc[f'< {STALE_THRESHOLD}d', 'decline_rate']
print(f"\nGap (stale minus fresh): {gap:+.3f}")

print("\nCONFOUND CHECK -- is the staleness effect just volume in disguise?")
print(frame.groupby(['vol_bucket', 'staleness_bucket'], observed=True)['y'].mean().unstack().round(3).to_string())
print("A staleness effect that survives inside every volume row is a real, separate signal;")
print("one that only shows up in the low-volume row is the same floor effect Signal 1 risks.")


Does the specific '180+ days stale' rule the refresh flag uses hold up?
                        n  decline_rate  median_daily_impr
days_since_update                                         
< 180d             116513         0.504             19.097
>= 180d                26         0.692              5.155

Gap (stale minus fresh): +0.188

CONFOUND CHECK -- is the staleness effect just volume in disguise?
staleness_bucket  freshest     Q2     Q3     Q4  stalest
vol_bucket                                              
Q1 low               0.546  0.356  0.487  0.357    0.381
Q2                   0.738  0.482  0.542  0.446    0.533
Q3                   0.721  0.517  0.511  0.480    0.570
Q4                   0.628  0.431  0.441  0.499    0.628
Q5 high              0.544  0.368  0.404  0.358    0.550
A staleness effect that survives inside every volume row is a real, separate signal;
one that only shows up in the low-volume row is the same floor effect Signal 1 risks.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [7]:
print(f"Position verdict:   {v1}")
print(f"Staleness verdict:  {v2}  (threshold test gap: {gap:+.3f})")
print(f"Volume verdict:     {v3}  (Spearman rho: {rho:.3f})")
print()
print("Read the three verdicts above before trusting the paragraph below -- it is written to")
print("describe whichever pattern they actually show, not to assume one in advance.")


Position verdict:   OPPOSITE
Staleness verdict:  OPPOSITE  (threshold test gap: +0.188)
Volume verdict:     OPPOSITE  (Spearman rho: 0.003)

Read the three verdicts above before trusting the paragraph below -- it is written to
describe whichever pattern they actually show, not to assume one in advance.


**For a content team:** the verdicts above decide whether an easy, popular assumption survives contact with this data. A `CONFIRMED` staleness signal means the existing refresh-flag logic is doing real work and is worth keeping as-is; an `OPPOSITE` or `FALSE` verdict means the team is spending review slots on a rule that doesn't track actual decline, and staleness should be demoted from "the reason to refresh" to "one weak tiebreaker among several." Either way, the confound check above is the part that matters most operationally — a staleness effect that only shows up among already-low-traffic pages isn't a reason to refresh anything; it's the floor effect Signal 1 already warned about, wearing a different column's name.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.